# An Autonomous Multi-Agent Deal Hunter
In this notebook, I built an autonomous multi-agent system that scans electronics deal feeds, estimates the true market value of each product, and sends push notifications when a significant discount is found.

The system coordinates four specialised agents: a **Scanner Agent** that scrapes DealNews RSS feeds and uses GPT-4o-mini with structured outputs to select the 5 most promising deals, a **Pricer Agent** that estimates each product's typical retail value, a **Planning Agent** that compares deal price against estimated value and identifies the best discount, and a **Messaging Agent** that delivers alerts via Pushover push notification when a discount exceeds the threshold.

The Gradio UI streams live agent logs in real time, displays all discovered deals in a sortable table, and supports both automatic 5-minute scans and manual runs. Clicking any row in the table triggers a manual Pushover alert for that deal.

#### Resources
- Dataset source: [DealNews](https://www.dealnews.com) RSS feeds (Electronics, Computers, Smart Home)
- Notifications: [Pushover](https://pushover.net)
- Models used: `gpt-4o-mini` (scanning, pricing)

In [2]:
# imports

import os
import json
import re
import time
import queue
import threading
import logging
import requests
import feedparser
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
import gradio as gr

In [36]:
# constants 

MODEL_GPT         = "gpt-4o-mini"
PUSHOVER_URL      = "https://api.pushover.net/1/messages.json"
MEMORY_FILENAME   = "memory.json"
DEAL_THRESHOLD    = 50        # minimum discount (USD) to trigger a Pushover alert

RSS_FEEDS = [
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c39/Computers/?rss=1",
    "https://www.dealnews.com/f1912/Smart-Home/?rss=1",
]

SYSTEM_PROMPT_SCANNER = """You identify and summarize the 5 most detailed deals from a list, by selecting deals
that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number
derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price.
It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products described as "$XXX off" or "reduced by $XXX" — this isn't the actual price.
Only respond with products when you are highly confident about the price."""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have
the most detailed, high quality product description and a clear price greater than 0.
Rephrase the description to summarize the product itself, not the terms of the deal.
Remember to respond with a short paragraph in the product_description field for each item.
Be careful with products described as "$XXX off" — this isn't the actual price.

Deals:

"""
USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [ ]:
# set up environment

load_dotenv(override=True)

openai_api_key    = os.getenv("OPENAI_API_KEY")
pushover_user     = os.getenv("PUSHOVER_USER")
pushover_token    = os.getenv("PUSHOVER_TOKEN")

print("OPENAI_API_KEY:  ", "✅ Found" if openai_api_key  else "❌ Missing")
print("PUSHOVER_USER:   ", "✅ Found" if pushover_user   else "❌ Missing")
print("PUSHOVER_TOKEN:  ", "✅ Found" if pushover_token  else "❌ Missing")

client = OpenAI() if openai_api_key else None

In [4]:
# Pydantic Models

class Deal(BaseModel):
    product_description: str = Field(
        description="A clearly expressed summary of the product in 3-4 sentences. "
                    "Focus on the item itself, not the discount terms."
    )
    price: float = Field(
        description="The actual price of this product as advertised."
    )
    url: str = Field(description="The URL of the deal.")


class DealSelection(BaseModel):
    deals: List[Deal] = Field(
        description="The 5 deals with the most detailed description and clearest price."
    )


In [5]:
# Memory Helpers

def read_memory() -> List[dict]:
    if os.path.exists(MEMORY_FILENAME):
        with open(MEMORY_FILENAME, "r") as f:
            return json.load(f)
    return []


def write_memory(memory: List[dict]) -> None:
    with open(MEMORY_FILENAME, "w") as f:
        json.dump(memory, f, indent=2)


def reset_memory() -> None:
    data = read_memory()
    write_memory(data[:2])
    print("Memory reset to 2 deals.")

In [6]:
# ScannerAgent

def scrape_deal(entry: dict) -> Optional[dict]:
    """Fetch a deal page and extract title, details, features and URL."""
    try:
        title = entry.get("title", "")[:100]
        url   = entry["links"][0]["href"]
        html  = requests.get(url, timeout=8).content
        soup  = BeautifulSoup(html, "html.parser")
        section = soup.find("div", class_="content-section")
        if not section:
            return None
        content = section.get_text().replace("\nmore", "").replace("\n", " ")
        if "Features" in content:
            details, features = content.split("Features", 1)
        else:
            details, features = content, ""
        return {
            "title":    title,
            "details":  details[:500],
            "features": features[:500],
            "url":      url,
        }
    except Exception:
        return None


def describe_deal(deal: dict) -> str:
    return (
        f"Title: {deal['title']}\n"
        f"Details: {deal['details'].strip()}\n"
        f"Features: {deal['features'].strip()}\n"
        f"URL: {deal['url']}"
    )


def fetch_deals(memory_urls: List[str]) -> List[dict]:
    """Scrape RSS feeds and return deals not already in memory."""
    logging.info("[Scanner Agent] Fetching deals from RSS feeds")
    raw = []
    for feed_url in RSS_FEEDS:
        feed = feedparser.parse(feed_url)
        for entry in feed.entries[:10]:
            deal = scrape_deal(entry)
            if deal and deal["url"] not in memory_urls:
                raw.append(deal)
            time.sleep(0.05)
    logging.info(f"[Scanner Agent] Found {len(raw)} new deals")
    return raw


def scan(memory_urls: List[str]) -> Optional[DealSelection]:
    """Ask GPT to select the 5 best deals using structured outputs."""
    scraped = fetch_deals(memory_urls)
    if not scraped:
        logging.info("[Scanner Agent] No new deals found")
        return None
    user_prompt = USER_PROMPT_PREFIX + "\n\n".join(describe_deal(d) for d in scraped) + USER_PROMPT_SUFFIX
    logging.info("[Scanner Agent] Calling GPT for deal selection")
    response = client.beta.chat.completions.parse(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_SCANNER},
            {"role": "user",   "content": user_prompt},
        ],
        response_format=DealSelection,
    )
    result = response.choices[0].message.parsed
    result.deals = [d for d in result.deals if d.price > 0]
    logging.info(f"[Scanner Agent] Selected {len(result.deals)} deals")
    return result

In [37]:
# PricerAgent

def estimate_price(description: str) -> float:
    """Use GPT-4o-mini to estimate the typical retail price of a product."""
    logging.info(f"[Pricer Agent] Estimating: {description[:50]}...")
    try:
        response = client.chat.completions.create(
            model=MODEL_GPT,
            messages=[
                {"role": "system", "content": "You are a product pricing expert. Given a product description, estimate its typical retail price in USD. Respond with just a number, no symbols or explanation."},
                {"role": "user", "content": description},
            ],
            max_tokens=10,
        )
        reply = (response.choices[0].message.content or "").replace("$", "").replace(",", "")
        match = re.search(r"[-+]?\d*\.?\d+", reply)
        result = float(match.group()) if match else 0.0
        logging.info(f"[Pricer Agent] Estimated value: ${result:.2f}")
        return result
    except Exception as e:
        logging.info(f"[Pricer Agent] Error estimating price: {e} — returning $0.00")
        return 0.0

In [38]:
# MessagingAgent

def push(message: str) -> None:
    """Send a Pushover push notification."""
    if not pushover_user or not pushover_token:
        logging.info("[Messaging Agent] Pushover not configured — skipping push")
        return
    logging.info("[Messaging Agent] Sending push notification")
    payload = {
        "user":    pushover_user,
        "token":   pushover_token,
        "message": message,
        "sound":   "cashregister",
    }
    requests.post(PUSHOVER_URL, data=payload, timeout=5)


def alert(deal: Deal, estimate: float, discount: float) -> None:
    text = (
        f"Deal Alert! Price=${deal.price:.2f}, "
        f"Estimate=${estimate:.2f}, Discount=${discount:.2f}\n"
        f"{deal.product_description[:80]}...\n"
        f"{deal.url}"
    )
    push(text)
    logging.info("[Messaging Agent] Alert sent")

In [39]:
# PlanningAgent

def plan(memory: List[dict]) -> Optional[dict]:
    """
    Full agent run:
    1. Scan RSS feeds for new deals
    2. Price each deal with the fine-tuned model
    3. Pick the best discount
    4. Alert via Pushover if discount exceeds threshold
    5. Return the best opportunity dict
    """
    logging.info("[Planning Agent] Starting run")
    memory_urls = [m["url"] for m in memory]
    selection = scan(memory_urls)
    if not selection:
        logging.info("[Planning Agent] No deals to process")
        return None

    best_opportunity = None
    best_discount    = 0.0

    for i, deal in enumerate(selection.deals):
        logging.info(f"[Planning Agent] Pricing deal {i+1}/{len(selection.deals)}: {deal.product_description[:50]}...")
        estimate = estimate_price(deal.product_description)
        discount = estimate - deal.price
        logging.info(f"[Planning Agent] Deal {i+1} — price=${deal.price:.2f}, estimate=${estimate:.2f}, discount=${discount:.2f}")
        if discount > best_discount:
            best_discount    = discount
            best_opportunity = {"description": deal.product_description, "price": deal.price,
                                "estimate": estimate, "discount": discount, "url": deal.url}

    if best_opportunity:
        logging.info(f"[Planning Agent] Best discount: ${best_discount:.2f}")
        if best_discount > DEAL_THRESHOLD:
            alert(
                Deal(product_description=best_opportunity["description"],
                     price=best_opportunity["price"],
                     url=best_opportunity["url"]),
                best_opportunity["estimate"],
                best_opportunity["discount"],
            )

    logging.info("[Planning Agent] Run complete")
    return best_opportunity

In [40]:
# Logging Helpers

class QueueHandler(logging.Handler):
    def __init__(self, log_queue):
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        self.log_queue.put(self.format(record))


def html_for(log_data: List[str]) -> str:
    output = "<br>".join(log_data[-18:])
    return (
        '<div style="height:400px;overflow-y:auto;border:1px solid #ccc;'
        'background-color:#222229;padding:10px;font-family:monospace;font-size:0.8rem;">'
        + output + "</div>"
    )


def setup_logging(log_queue):
    handler = QueueHandler(log_queue)
    handler.setFormatter(logging.Formatter("[%(asctime)s] %(message)s", datefmt="%H:%M:%S"))
    logger = logging.getLogger()
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

In [49]:
# Gradio UI

memory = read_memory()


def table_for(mem: List[dict]) -> List[List]:
    return [
        [m["description"], f"${m['price']:.2f}", f"${m['estimate']:.2f}", f"${m['discount']:.2f}", m["url"]]
        for m in mem
    ]


COLORS = {
    "[Scanner Agent]":  "#00dddd",
    "[Pricer Agent]":   "#dd0000",
    "[Planning Agent]": "#00dd00",
    "[Messaging Agent]":"#87CEEB",
}

def colorize(msg: str) -> str:
    for tag, color in COLORS.items():
        if tag in msg:
            return f'<span style="color:{color}">{msg}</span>'
    return f'<span style="color:#aaaaaa">{msg}</span>'


def run_with_logging(log_data: List[str]):
    global memory
    log_queue    = queue.Queue()
    result_queue = queue.Queue()
    setup_logging(log_queue)

    def worker():
        result = plan(memory)
        if result:
            memory.append(result)
            write_memory(memory)
        result_queue.put(table_for(memory))

    threading.Thread(target=worker).start()

    final_result = None
    worker_done  = False
    while not worker_done:
        # drain all pending log messages
        drained = False
        while True:
            try:
                msg = log_queue.get_nowait()
                log_data.append(colorize(msg))
                drained = True
            except queue.Empty:
                break
        # check if worker finished
        try:
            final_result = result_queue.get_nowait()
            worker_done  = True
        except queue.Empty:
            pass
        if drained or worker_done:
            yield log_data, html_for(log_data), final_result or table_for(memory)
        else:
            time.sleep(0.1)


def do_select(selected_index: gr.SelectData):
    if selected_index.index[0] < len(memory):
        opp = memory[selected_index.index[0]]
        push(
            f"Manual alert: {opp['description'][:80]}...\n"
            f"Price=${opp['price']:.2f} Estimate=${opp['estimate']:.2f}\n{opp['url']}"
        )


welcome = [[
    "Welcome! I'm your deal-hunting assistant. I scan electronics and tech RSS feeds, "
    "estimate each product's true value using a fine-tuned GPT model, and alert you when "
    "I find a significant discount. Deals will appear here automatically. "
    "Click any row to send yourself a Pushover alert for that deal.",
    "", "", "", ""
]]

with gr.Blocks(title="Deal Hunter") as ui:
    log_data = gr.State([])

    gr.Markdown(
        '<div style="text-align:center;font-size:1.5rem;font-weight:700;margin-bottom:4px">'
        'Deal Hunter — Autonomous Multi-Agent Deal Scanner</div>'
    )
    gr.Markdown(
        '<div style="text-align:center;font-size:0.9rem;color:#888;margin-bottom:12px">'
        'Scans electronics RSS feeds every 5 minutes · Prices deals with a fine-tuned GPT model · '
        'Sends Pushover alerts for the best discounts</div>'
    )

    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Deal Price", "Estimated Value", "Discount", "URL"],
            value=welcome,
            wrap=True,
            column_widths=[6, 1, 1, 1, 3],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("**Agent Logs**")
            logs = gr.HTML(html_for([]))
        with gr.Column(scale=1):
            gr.Markdown("**How it works**")
            gr.Markdown(
                "1. **Scanner Agent** — fetches deals from DealNews RSS feeds and uses "
                "GPT-4o-mini with structured outputs to select the 5 best.\n"
                "2. **Pricer Agent** — calls GPT-4o-mini to estimate each product's "
                "typical retail value.\n"
                "3. **Planning Agent** — compares deal price vs estimated value, picks "
                "the best discount, and decides whether to alert.\n"
                "4. **Messaging Agent** — sends a Pushover push notification when a "
                f"discount exceeds ${DEAL_THRESHOLD}.\n\n"
                "Click any row in the table to manually trigger a Pushover alert."
            )

    with gr.Row():
        run_btn = gr.Button("▶ Run Now", variant="primary")

    run_btn.click(fn=lambda: gr.update(interactive=False), outputs=run_btn).then(
        fn=run_with_logging, inputs=[log_data], outputs=[log_data, logs, opportunities_dataframe]
    ).then(fn=lambda: gr.update(interactive=True), outputs=run_btn)

    ui.load(lambda: ([], html_for([]), table_for(memory)), inputs=[], outputs=[log_data, logs, opportunities_dataframe])

    timer = gr.Timer(value=300, active=True)
    timer.tick(run_with_logging, inputs=[log_data], outputs=[log_data, logs, opportunities_dataframe])
    opportunities_dataframe.select(do_select)

In [ ]:
# Launch
ui.launch(inbrowser=True)